# Hidden State Extraction — Multiple Designs

Extracts hidden states from Llama-3.1-8B-Instruct using six different extraction strategies.
Run one folder at a time (change `FOLDER_NAME` in the config cell). Each design saves to its own subfolder.

## Design reference

| Design | Token position | Context | Turns extracted | Research question |
|--------|---------------|---------|-----------------|-------------------|
| **A** | Last token of each **assistant** response | Full context up to turn t (current design) | All turns | How does model state evolve as it generates each response? |
| **B** | Last token of **final** assistant response | Varies: k = 1 … n prior turns in context | Final turn only (×n k-values) | Does adding prior context shift the final-turn representation? Is the shift larger for jailbroken vs refusal? |
| **C** | Last token of each **user** message | Full context up to end of user turn t | All turns | Does the attacker's framing encode escalation before the model generates? |
| **D** | **First** token of each assistant response | Full context up to turn t | All turns | Is compliance/refusal committed at the first generated token? |
| **E** | **Mean pool** across all tokens of each assistant response | Full context up to turn t | All turns | Does averaging over the full response give a more stable stance representation? |
| **F** | Last token of **final** assistant response | Full context but prior assistant responses replaced with placeholder | Final turn only | Are the model's own prior responses the active ingredient, or is it the user-turn framing? |

**Output:** `data/representations/{FOLDER_NAME}_design_{X}/hidden_states.npy` + `metadata.parquet`

Designs A, C, D, E share one forward pass per conversation (efficient).  
Design B requires n forward passes per conversation (one per k value — expensive).  
Design F requires one additional forward pass per conversation.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── Config ────────────────────────────────────────────────────────────────────
FOLDER_NAME = "actorattack_harmful"   # ← change this to process a different folder

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
DEVICE   = "cuda:0"
DTYPE    = torch.bfloat16

CONV_DIR  = repo_root / "data" / "conversations" / FOLDER_NAME
REPR_ROOT = repo_root / "data" / "representations"

parts     = FOLDER_NAME.split("_")
FRAMEWORK = parts[0]
SPLIT     = parts[1]

assert CONV_DIR.exists(), f"Conversation folder not found: {CONV_DIR}"
print(f"Folder:    {FOLDER_NAME}")
print(f"Framework: {FRAMEWORK}")
print(f"Split:     {SPLIT}")
print(f"Input:     {CONV_DIR}")

In [ ]:
print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map=DEVICE,
)
model.eval()
print(f"Loaded. Layers: {model.config.num_hidden_layers}, Hidden dim: {model.config.hidden_size}")

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────

def build_context(turns, framework):
    """Build filtered messages list and extract_at entries. Skips is_refusal turns for Crescendo."""
    messages, extract_at = [], []
    by_idx = {}
    for t in turns:
        by_idx.setdefault(t["turn_idx"], {})[t["role"]] = t
    for turn_idx in sorted(by_idx):
        pair = by_idx[turn_idx]
        user_turn = pair.get("user")
        asst_turn = pair.get("assistant")
        if not user_turn or not asst_turn:
            continue
        if framework == "crescendo" and asst_turn.get("is_refusal", False):
            continue
        messages.append({"role": "user",      "content": user_turn["content"]})
        messages.append({"role": "assistant", "content": asst_turn["content"]})
        extract_at.append({"turn_idx": turn_idx, "n_messages": len(messages)})
    return messages, extract_at


def save_design(states_list, meta_list, save_dir):
    """Concatenate and save hidden states and metadata."""
    save_dir.mkdir(parents=True, exist_ok=True)
    states = np.concatenate(states_list, axis=0).astype(np.float16)
    meta   = pd.concat(meta_list, ignore_index=True)
    np.save(str(save_dir / "hidden_states.npy"), states)
    meta.to_parquet(save_dir / "metadata.parquet", index=False)
    print(f"  Saved {states.shape[0]} vectors → {save_dir}")
    print(f"  Shape: {states.shape}  ({states.nbytes / 1e6:.1f} MB)")
    return states, meta


def load_done(save_dir):
    """Return set of already-processed (pair_id, attempt) pairs for resume support."""
    meta_path = save_dir / "metadata.parquet"
    if meta_path.exists():
        existing = pd.read_parquet(meta_path)
        done = set(zip(existing["pair_id"], existing["attempt"]))
        print(f"  Resuming: {len(done)} conversations already done")
        states_existing = [np.load(str(save_dir / "hidden_states.npy"))]
        return done, states_existing, [existing]
    return set(), [], []


def base_row(conv, fpath, framework, split):
    return dict(
        framework = framework,
        split     = split,
        pair_id   = conv["objective_pair_id"],
        attempt   = conv.get("attempt", 1),
        verdict   = conv["verdict"],
        label     = 1 if split == "harmful" else 0,
        fname     = fpath.name,
    )


files = sorted(CONV_DIR.glob("*.json"))
print(f"Files to process: {len(files)}")
print("Helpers defined.")

## Design A — Last token of each assistant response (current design)

Extracts at the last token of each assistant turn's response, with full causal context up to that turn.
One forward pass per conversation; positions found via prefix tokenization.

In [ ]:
SAVE_A = REPR_ROOT / f"{FOLDER_NAME}_design_A"
done_a, all_states_a, all_meta_a = load_done(SAVE_A)

for fpath in tqdm(files, desc="Design A"):
    conv    = json.loads(fpath.read_text())
    pair_id = conv["objective_pair_id"]
    attempt = conv.get("attempt", 1)
    if (pair_id, attempt) in done_a:
        continue
    turns = conv.get("turns", [])
    if not turns:
        continue
    try:
        messages, extract_at = build_context(turns, FRAMEWORK)
        if not extract_at:
            continue

        input_ids = tokenizer.apply_chat_template(
            messages, return_tensors="pt", add_generation_prompt=False
        ).to(model.device)

        with torch.no_grad():
            outputs = model(input_ids, output_hidden_states=True)
        last_hidden = outputs.hidden_states[-1][0]  # (seq_len, hidden_dim)

        vecs, rows = [], []
        for entry in extract_at:
            prefix_ids = tokenizer.apply_chat_template(
                messages[:entry["n_messages"]], return_tensors="pt", add_generation_prompt=False
            )
            pos = prefix_ids.shape[1] - 1
            vecs.append(last_hidden[pos].cpu().to(torch.float16).numpy())
            rows.append({**base_row(conv, fpath, FRAMEWORK, SPLIT), "turn_idx": entry["turn_idx"]})

        all_states_a.append(np.stack(vecs))
        all_meta_a.append(pd.DataFrame(rows))
        done_a.add((pair_id, attempt))
    except Exception as e:
        tqdm.write(f"ERROR {fpath.name}: {e}")

save_design(all_states_a, all_meta_a, SAVE_A)

## Design B — Last token of final assistant response, varying prior context (Bullwinkel Design A)

Always extracts at the last token of the **final** assistant response.
Varies k = number of prior turns in context: k=1 (final turn only) → k=n (full conversation).

**Expensive:** n forward passes per conversation. ~10× slower than other designs.

In [ ]:
SAVE_B = REPR_ROOT / f"{FOLDER_NAME}_design_B"

# Resume keyed on (pair_id, attempt, k)
meta_path_b = SAVE_B / "metadata.parquet"
if meta_path_b.exists():
    existing_b = pd.read_parquet(meta_path_b)
    done_b = set(zip(existing_b["pair_id"], existing_b["attempt"], existing_b["k"]))
    all_states_b = [np.load(str(SAVE_B / "hidden_states.npy"))]
    all_meta_b   = [existing_b]
    print(f"  Resuming: {len(done_b)} (pair_id, attempt, k) triples already done")
else:
    done_b, all_states_b, all_meta_b = set(), [], []

for fpath in tqdm(files, desc="Design B"):
    conv    = json.loads(fpath.read_text())
    pair_id = conv["objective_pair_id"]
    attempt = conv.get("attempt", 1)
    turns   = conv.get("turns", [])
    if not turns:
        continue
    try:
        messages, extract_at = build_context(turns, FRAMEWORK)
        if not extract_at:
            continue
        n_turns = len(messages) // 2  # total non-refusal turns

        vecs, rows = [], []
        for k in range(1, n_turns + 1):
            if (pair_id, attempt, k) in done_b:
                continue
            # Context = last k turns of the filtered message list
            context = messages[-(2 * k):]
            input_ids = tokenizer.apply_chat_template(
                context, return_tensors="pt", add_generation_prompt=False
            ).to(model.device)
            with torch.no_grad():
                outputs = model(input_ids, output_hidden_states=True)
            last_hidden = outputs.hidden_states[-1][0]
            pos = input_ids.shape[1] - 1
            vecs.append(last_hidden[pos].cpu().to(torch.float16).numpy())
            rows.append({
                **base_row(conv, fpath, FRAMEWORK, SPLIT),
                "k":        k,
                "n_turns":  n_turns,
                "turn_idx": extract_at[-1]["turn_idx"],  # final turn index
            })

        if vecs:
            all_states_b.append(np.stack(vecs))
            all_meta_b.append(pd.DataFrame(rows))
            for row in rows:
                done_b.add((pair_id, attempt, row["k"]))
    except Exception as e:
        tqdm.write(f"ERROR {fpath.name}: {e}")

save_design(all_states_b, all_meta_b, SAVE_B)

## Design C — Last token of each user message

Extracts at the last token of the user's question at each turn (including the EOT token).
Captures the model's representation of the attacker's framing before it starts generating.

In [ ]:
SAVE_C = REPR_ROOT / f"{FOLDER_NAME}_design_C"
done_c, all_states_c, all_meta_c = load_done(SAVE_C)

for fpath in tqdm(files, desc="Design C"):
    conv    = json.loads(fpath.read_text())
    pair_id = conv["objective_pair_id"]
    attempt = conv.get("attempt", 1)
    if (pair_id, attempt) in done_c:
        continue
    turns = conv.get("turns", [])
    if not turns:
        continue
    try:
        messages, extract_at = build_context(turns, FRAMEWORK)
        if not extract_at:
            continue

        input_ids = tokenizer.apply_chat_template(
            messages, return_tensors="pt", add_generation_prompt=False
        ).to(model.device)
        with torch.no_grad():
            outputs = model(input_ids, output_hidden_states=True)
        last_hidden = outputs.hidden_states[-1][0]

        vecs, rows = [], []
        for entry in extract_at:
            # User prefix: all messages up to and including user turn t (no assistant response)
            user_prefix = messages[:entry["n_messages"] - 1]
            user_ids = tokenizer.apply_chat_template(
                user_prefix, return_tensors="pt", add_generation_prompt=False
            )
            pos = user_ids.shape[1] - 1  # last token of user message (incl. EOT)
            vecs.append(last_hidden[pos].cpu().to(torch.float16).numpy())
            rows.append({**base_row(conv, fpath, FRAMEWORK, SPLIT), "turn_idx": entry["turn_idx"]})

        all_states_c.append(np.stack(vecs))
        all_meta_c.append(pd.DataFrame(rows))
        done_c.add((pair_id, attempt))
    except Exception as e:
        tqdm.write(f"ERROR {fpath.name}: {e}")

save_design(all_states_c, all_meta_c, SAVE_C)

## Design D — First token of each assistant response

Extracts at the first generated token of each assistant response.
The first token is the model's most decisive moment — it has processed full context and commits to a direction.

In [ ]:
SAVE_D = REPR_ROOT / f"{FOLDER_NAME}_design_D"
done_d, all_states_d, all_meta_d = load_done(SAVE_D)

for fpath in tqdm(files, desc="Design D"):
    conv    = json.loads(fpath.read_text())
    pair_id = conv["objective_pair_id"]
    attempt = conv.get("attempt", 1)
    if (pair_id, attempt) in done_d:
        continue
    turns = conv.get("turns", [])
    if not turns:
        continue
    try:
        messages, extract_at = build_context(turns, FRAMEWORK)
        if not extract_at:
            continue

        input_ids = tokenizer.apply_chat_template(
            messages, return_tensors="pt", add_generation_prompt=False
        ).to(model.device)
        with torch.no_grad():
            outputs = model(input_ids, output_hidden_states=True)
        last_hidden = outputs.hidden_states[-1][0]

        vecs, rows = [], []
        for entry in extract_at:
            # Generation prompt prefix: up to user turn t, with add_generation_prompt=True
            # This gives the length of everything before the first assistant content token
            user_prefix = messages[:entry["n_messages"] - 1]
            gen_ids = tokenizer.apply_chat_template(
                user_prefix, return_tensors="pt", add_generation_prompt=True
            )
            # First assistant content token position in the full sequence
            pos_first = gen_ids.shape[1]
            # Last assistant token (for bounds check)
            full_ids = tokenizer.apply_chat_template(
                messages[:entry["n_messages"]], return_tensors="pt", add_generation_prompt=False
            )
            pos_last = full_ids.shape[1] - 1
            pos_first = min(pos_first, pos_last)  # clamp in case of empty response

            vecs.append(last_hidden[pos_first].cpu().to(torch.float16).numpy())
            rows.append({**base_row(conv, fpath, FRAMEWORK, SPLIT), "turn_idx": entry["turn_idx"]})

        all_states_d.append(np.stack(vecs))
        all_meta_d.append(pd.DataFrame(rows))
        done_d.add((pair_id, attempt))
    except Exception as e:
        tqdm.write(f"ERROR {fpath.name}: {e}")

save_design(all_states_d, all_meta_d, SAVE_D)

## Design E — Mean pool across all assistant response tokens

Averages hidden states across all token positions of each assistant response.
Less sensitive to position-specific effects than the single last or first token.

In [ ]:
SAVE_E = REPR_ROOT / f"{FOLDER_NAME}_design_E"
done_e, all_states_e, all_meta_e = load_done(SAVE_E)

for fpath in tqdm(files, desc="Design E"):
    conv    = json.loads(fpath.read_text())
    pair_id = conv["objective_pair_id"]
    attempt = conv.get("attempt", 1)
    if (pair_id, attempt) in done_e:
        continue
    turns = conv.get("turns", [])
    if not turns:
        continue
    try:
        messages, extract_at = build_context(turns, FRAMEWORK)
        if not extract_at:
            continue

        input_ids = tokenizer.apply_chat_template(
            messages, return_tensors="pt", add_generation_prompt=False
        ).to(model.device)
        with torch.no_grad():
            outputs = model(input_ids, output_hidden_states=True)
        last_hidden = outputs.hidden_states[-1][0]

        vecs, rows = [], []
        for entry in extract_at:
            user_prefix = messages[:entry["n_messages"] - 1]
            full_prefix = messages[:entry["n_messages"]]

            gen_ids  = tokenizer.apply_chat_template(
                user_prefix, return_tensors="pt", add_generation_prompt=True
            )
            full_ids = tokenizer.apply_chat_template(
                full_prefix, return_tensors="pt", add_generation_prompt=False
            )

            pos_start = gen_ids.shape[1]       # first assistant content token
            pos_end   = full_ids.shape[1]      # one past last assistant token

            if pos_start < pos_end:
                mean_vec = last_hidden[pos_start:pos_end].mean(dim=0)
            else:
                mean_vec = last_hidden[pos_end - 1]  # fallback: last token

            vecs.append(mean_vec.cpu().to(torch.float16).numpy())
            rows.append({**base_row(conv, fpath, FRAMEWORK, SPLIT), "turn_idx": entry["turn_idx"]})

        all_states_e.append(np.stack(vecs))
        all_meta_e.append(pd.DataFrame(rows))
        done_e.add((pair_id, attempt))
    except Exception as e:
        tqdm.write(f"ERROR {fpath.name}: {e}")

save_design(all_states_e, all_meta_e, SAVE_E)

## Design F — Final turn, prior assistant responses masked

Extracts at the last token of the **final** assistant response.
Prior assistant responses are replaced with a single placeholder token `"."` —
only the user-turn framing accumulates in context.

Compares to Design B at k=n (full context) to isolate whether the model's
own prior responses are the active ingredient driving compliance.

In [ ]:
SAVE_F = REPR_ROOT / f"{FOLDER_NAME}_design_F"
done_f, all_states_f, all_meta_f = load_done(SAVE_F)

PLACEHOLDER = "."  # replaces prior assistant response content

def mask_prior_assistant(messages):
    """Replace all assistant responses except the final one with PLACEHOLDER."""
    masked = []
    asst_positions = [i for i, m in enumerate(messages) if m["role"] == "assistant"]
    last_asst_idx  = asst_positions[-1] if asst_positions else -1
    for i, msg in enumerate(messages):
        if msg["role"] == "assistant" and i != last_asst_idx:
            masked.append({"role": "assistant", "content": PLACEHOLDER})
        else:
            masked.append(msg)
    return masked


for fpath in tqdm(files, desc="Design F"):
    conv    = json.loads(fpath.read_text())
    pair_id = conv["objective_pair_id"]
    attempt = conv.get("attempt", 1)
    if (pair_id, attempt) in done_f:
        continue
    turns = conv.get("turns", [])
    if not turns:
        continue
    try:
        messages, extract_at = build_context(turns, FRAMEWORK)
        if not extract_at:
            continue

        masked_messages = mask_prior_assistant(messages)

        input_ids = tokenizer.apply_chat_template(
            masked_messages, return_tensors="pt", add_generation_prompt=False
        ).to(model.device)
        with torch.no_grad():
            outputs = model(input_ids, output_hidden_states=True)
        last_hidden = outputs.hidden_states[-1][0]

        # Extract only at the final turn
        final_entry = extract_at[-1]
        pos = input_ids.shape[1] - 1
        vec = last_hidden[pos].cpu().to(torch.float16).numpy()

        all_states_f.append(vec[np.newaxis, :])  # shape (1, hidden_dim)
        all_meta_f.append(pd.DataFrame([{
            **base_row(conv, fpath, FRAMEWORK, SPLIT),
            "turn_idx": final_entry["turn_idx"],
            "n_turns":  len(extract_at),
        }]))
        done_f.add((pair_id, attempt))
    except Exception as e:
        tqdm.write(f"ERROR {fpath.name}: {e}")

save_design(all_states_f, all_meta_f, SAVE_F)